In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

if os.path.exists('notebooks'):
    os.chdir('notebooks')

# Titanic Survivability

Predict survival on the Titanic and get familiar with ML basics.

## Initialization

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import utils.transformers as trf

from ml_utils import utils
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [4]:
df_train = pd.read_parquet("../data/processed/X_train.parquet", engine='fastparquet')
df_test = pd.read_parquet("../data/processed/X_test.parquet", engine='fastparquet')
df_train_y = pd.read_parquet("../data/processed/y_train.parquet", engine='fastparquet')
df_test_y = pd.read_parquet("../data/processed/y_test.parquet", engine='fastparquet')

In [5]:
utils.skim_data(df_train)

Total duplicate rows: 0
DF shape: (712, 11)


,feature,dtype,null_%,negative_%,zero_%,min,max,n_unique,unique_%,sample_values
0,PassengerId,int64,0.000,0.0,0.0,2,891,712,100.00,"[285, 862, 191, 44, 742]"
1,Pclass,category,0.000,-,-,-,-,3,0.42,"[1, 2, 3]"
2,Name,string,0.000,-,-,-,-,712,100.00,"[Smith, Mr. Richard William, Giles, Mr. Frederick Edward, Pinsky, Mrs. (Rosa), Laroche, Miss. Simonne Marie Anne Andree, Cavendish, Mr. Tyrell William]"
3,Sex,category,0.000,-,-,-,-,2,0.28,"[male, female]"
4,Age,float64,19.242,0.0,0.0,0.42,80.0,81,11.38,"[21.0, 32.0, 3.0, 36.0, 28.0]"
5,SibSp,int64,0.000,0.0,68.399,0,8,7,0.98,"[0, 1, 3, 4, 2]"
6,Parch,int64,0.000,0.0,75.983,0,6,7,0.98,"[0, 2, 1, 4, 3]"
7,Ticket,string,0.000,-,-,-,-,571,80.20,"[113056, 28134, 234604, SC/Paris 2123, 19877]"
8,Fare,float64,0.000,0.0,1.826,0.0,512.3292,222,31.18,"[26.0, 11.5, 13.0, 41.5792, 78.85]"
9,Cabin,string,77.528,-,-,-,-,119,16.71,"[A19, C46, B78, C93, C95]"


## Feature Engineering

Features we engineered so far:

In [6]:
feature_engineering_stage = Pipeline([
    ('title_generator', trf.TitleTransformer()),
    ('age_imputer', trf.AgeImputer()),
    ('cabin_indicator', trf.CabinIndicatorTransformer()),
    ('embarked_imputer', trf.EmbarkedImputer()),
    ('auto_skewness', trf.AutoSkewnessTransformer(threshold=0.5))
])
feature_engineering_stage.set_output(transform='pandas')

Pipeline(steps=[('title_generator', TitleTransformer()),
                ('age_imputer', AgeImputer()),
                ('cabin_indicator', CabinIndicatorTransformer()),
                ('embarked_imputer', EmbarkedImputer()),
                ('auto_skewness', AutoSkewnessTransformer())])

In [9]:
df_temp = feature_engineering_stage.fit_transform(df_train)
utils.skim_data(df_temp) # type: ignore

Total duplicate rows: 0
DF shape: (712, 12)


,feature,dtype,null_%,negative_%,zero_%,min,max,n_unique,unique_%,sample_values
0,PassengerId,int64,0.0,0.0,0.0,2,891,712,100.00,"[285, 862, 191, 44, 742]"
1,Pclass,category,0.0,-,-,-,-,3,0.42,"[1, 2, 3]"
2,Name,string,0.0,-,-,-,-,712,100.00,"[Smith, Mr. Richard William, Giles, Mr. Frederick Edward, Pinsky, Mrs. (Rosa), Laroche, Miss. Simonne Marie Anne Andree, Cavendish, Mr. Tyrell William]"
3,Sex,category,0.0,-,-,-,-,2,0.28,"[male, female]"
4,Age,float64,0.0,0.0,0.0,0.350657,4.394449,81,11.38,"[3.4011973816621555, 3.091042453358316, 3.4965075614664802, 1.3862943611198906, 3.6109179126442243]"
5,SibSp,float64,0.0,0.0,68.399,0.0,2.197225,7,0.98,"[0.0, 0.6931471805599453, 1.3862943611198906, 1.6094379124341003, 1.0986122886681098]"
6,Parch,float64,0.0,0.0,75.983,0.0,1.94591,7,0.98,"[0.0, 1.0986122886681098, 0.6931471805599453, 1.6094379124341003, 1.3862943611198906]"
7,Ticket,string,0.0,-,-,-,-,571,80.20,"[113056, 28134, 234604, SC/Paris 2123, 19877]"
8,Fare,float64,0.0,0.0,1.826,0.0,6.240917,222,31.18,"[3.295836866004329, 2.5257286443082556, 2.6390573296152584, 3.7513658711253766, 4.380149874661021]"
9,Embarked,category,0.0,-,-,-,-,3,0.42,"[S, C, Q]"


## Preprocessing

In [12]:
preprocessing_stage = ColumnTransformer(transformers=[
    ('numeric_scaler', StandardScaler(), ['Age', 'Fare', 'SibSp', 'Parch']),
    (
        'categorical_encoder',
        OneHotEncoder(handle_unknown='ignore', sparse_output=False), 
        ['Pclass', 'Sex', 'Embarked', 'Title']
    ),
    ('binary_passthrough', 'passthrough', ['HasCabin'])
])

full_pipeline = Pipeline([
    ('engineering', feature_engineering_stage),
    ('processing', preprocessing_stage)
])

full_pipeline.set_output(transform="pandas")

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('engineering', ...), ('processing', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('title_generator', ...), ('age_imputer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,rare_threshold_reference,'Master'
,drop_original,True
,threshold,0.5


In [13]:
df_temp = full_pipeline.fit_transform(df_train)
utils.skim_data(df_temp)

Total duplicate rows: 87
DF shape: (712, 18)


,feature,dtype,null_%,negative_%,zero_%,min,max,n_unique,unique_%,sample_values
0,numeric_scaler__Age,float64,0.0,38.624,0.000,-4.813397,1.842963,81,11.38,"[0.2080022604694644, -0.30253401324311835, 0.3648893501258562, -3.108666236869843, 0.5532166312168477]"
1,numeric_scaler__Fare,float64,0.0,57.163,0.000,-3.058584,3.444390,222,31.18,"[0.37564522921595855, -0.4267998264257639, -0.3087124711360148, 0.8503018911792926, 1.505488526627153]"
2,numeric_scaler__SibSp,float64,0.0,68.399,0.000,-0.605763,4.124085,7,0.98,"[-0.6057625614176696, 0.8863383147930254, 2.3784391910037206, 2.858788383461994, 1.759161374669462]"
3,numeric_scaler__Parch,float64,0.0,75.983,0.000,-0.530268,4.168984,7,0.98,"[-0.5302680053945426, 2.1228125896901853, 1.1436394806674803, 3.356424814735144, 2.817546966729503]"
4,categorical_encoder__Pclass_1,float64,0.0,0.000,76.264,0.000000,1.000000,2,0.28,"[1.0, 0.0]"
5,categorical_encoder__Pclass_2,float64,0.0,0.000,79.354,0.000000,1.000000,2,0.28,"[0.0, 1.0]"
6,categorical_encoder__Pclass_3,float64,0.0,0.000,44.382,0.000000,1.000000,2,0.28,"[0.0, 1.0]"
7,categorical_encoder__Sex_female,float64,0.0,0.000,64.888,0.000000,1.000000,2,0.28,"[0.0, 1.0]"
8,categorical_encoder__Sex_male,float64,0.0,0.000,35.112,0.000000,1.000000,2,0.28,"[1.0, 0.0]"
9,categorical_encoder__Embarked_C,float64,0.0,0.000,81.882,0.000000,1.000000,2,0.28,"[0.0, 1.0]"


---

In [14]:
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'kaggle_secrets' in sys.modules or os.path.exists('/kaggle')
IN_CLOUD = IN_COLAB or IN_KAGGLE

if IN_CLOUD:
    print("☁️ Cloud environment detected. Preparing to push changes to GitHub...")
    GIT_EMAIL = "dev.rbennum@gmail.com"
    GIT_NAME = "Bening R."
    COMMIT_MESSAGE = f"{""}"

    !git config --global user.email "{GIT_EMAIL}"
    !git config --global user.name "{GIT_NAME}"

    print("🔄 Staging files...")
    !git add .
    status = !git status --porcelain
    if status:
        print("📝 Committing changes...")
        !git commit -m "{COMMIT_MESSAGE}"
        print("🚀 Pushing to GitHub (Main Branch)...")
        !git push origin main
        print("✨ Success! Your cloud changes are now safely backed up on GitHub.")
    else:
        print("ℹ️ No changes detected in the notebooks. GitHub is already up to date.")
else:
    print("💻 Running locally on MacBook. Use your standard Git terminal or VS Code UI to push your work!")

💻 Running locally on MacBook. Use your standard Git terminal or VS Code UI to push your work!
